# Comparable CPET matrices: 4-minute and 8-minute synthetic ramps

**Version 0.2.1 — manuscript and software under peer review.** This notebook uses
invented observations only. It teaches the mapping; it is **not** a reproduction
of the manuscript reconstruction experiment or clinical validation.

Use **Runtime → Run all** in Google Colab. The final cell offers a ZIP download.
Local verification is documented in [VERIFICATION](https://github.com/mlacasa/CPET_method/blob/v0.2.1/VERIFICATION.md).
Status: **verificado localmente; ejecución en Colab pendiente**.

Input data must already be harmonised and quality controlled, with audited
phase markers. The package does not perform the full device ingestion or
physiological QC workflow. Public review access is provided; reuse license pending.

## 1. Install the identifiable review version

The normal installation uses the Git tag `v0.2.1`. Matplotlib is needed only for
plots. The environment override below is for maintainers verifying an unpublished
local build; readers in Colab need not set it.

In [ ]:
import os, subprocess, sys
VERSION = "0.2.1"
package_source = os.environ.get("CPET_TUTORIAL_PACKAGE", f"git+https://github.com/mlacasa/CPET_method.git@v{VERSION}")
subprocess.check_call([sys.executable, "-m", "pip", "install", package_source, "matplotlib>=3.7"])
from importlib.metadata import version
assert version("cpet-functional-skeleton") == VERSION
print("CPET package:", version("cpet-functional-skeleton"))

## 2. Generate inputs and understand their meaning

`time_s` is finite elapsed time in seconds, unique per native observation;
`workload_w` is finite delivered workload in watts; `VCO2` is a synthetic channel
in mL/min. NaNs represent unavailable channel observations, never zero.
Rows with channel gaps remain in place for the five-position moving mean.

The ramps last exactly **240 and 480 s**, start at 20% of peak, and reach **100 and
160 W**, respectively. Their load slopes are **20 and 16 W/min**. Recovery starts
2 s after each peak at 10 W and is observed for 180 s, using 61 and 91 native
positions. Ramp counts are 81 and 161. Sampling is deliberately irregular.

The known mathematical targets are `300 + 800*p/100` for ramp and
`300 + 800*exp(-tau/60)` for recovery. Both records share those coordinate-based
targets by construction, not because relative workload guarantees physiological
equivalence. At 50% of peak their absolute loads are 50 and 80 W.

Three separate illustrations:
1. **Alignment:** compare the four/eight-minute records against the same target.
   Default smoothing, binning and unequal sampling can yield nonzero differences.
2. **Known difference:** add exactly 30 mL/min to a copy of the eight-minute record,
   preserving its timestamps and missingness; isolate the amplitude change.
3. **Missing phase:** truncate another copy at peak; no recovery is inferred from
   another record. No random noise is added; seeds control sampling only.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json, shutil, tempfile, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from cpet_skeleton import PhaseMarkers, transform_record, compare_records
def analytic_response(coordinate, phase):
    """Invented VCO2 in mL/min; not fitted physiological ground truth."""
    x = np.asarray(coordinate, dtype=float)
    return 300 + 800 * (x / 100 if phase == 'ramp' else np.exp(-x / 60))

def synthetic_record(duration_s, peak_w, n_ramp, n_recovery, seed):
    rng = np.random.default_rng(seed)

    def irregular_times(duration, count):
        times = np.linspace(0, duration, count)
        times[1:-1] += rng.uniform(-.35, .35, count - 2) * duration / (count - 1)
        return times

    ramp_t = irregular_times(duration_s, n_ramp)
    tau = irregular_times(180, n_recovery)
    anchor = duration_s + 2
    # Start at 20% of the supplied Wpeak, reach Wpeak at the audited peak time.
    workload = peak_w * (.2 + .8 * ramp_t / duration_s)
    workload[-1] = peak_w
    ramp_y = analytic_response(100 * workload / peak_w, 'ramp')
    recovery_y = analytic_response(tau, 'recovery')
    # Deliberate internal and terminal channel gaps; retain the native rows.
    ramp_y[8::17] = np.nan
    recovery_y[8::17] = np.nan
    recovery_y[-2:] = np.nan
    frame = pd.DataFrame({
        'time_s': np.r_[ramp_t, anchor + tau],
        'workload_w': np.r_[workload, np.full(n_recovery, 10.)],
        'VCO2': np.r_[ramp_y, recovery_y],
    })
    return frame, PhaseMarkers(0., float(duration_s), float(peak_w), float(anchor))

def make_cases():
    short = synthetic_record(240, 100, 81, 61, 401)
    long = synthetic_record(480, 160, 161, 91, 801)
    shifted = long[0].copy()
    shifted['VCO2'] += 30  # Preserve coordinates and NaNs; isolate an amplitude change.
    absent = long[0].loc[long[0].time_s <= long[1].peak_time_s].copy()
    absent_markers = PhaseMarkers(0., 480., 160.)
    return {'ramp_4min': short, 'ramp_8min': long,
            'ramp_8min_plus30': (shifted, long[1]),
            'ramp_8min_no_recovery': (absent, absent_markers)}

cases = make_cases()
display(cases["ramp_4min"][0].head(10))
display(pd.DataFrame([dict(record=name, **asdict(markers)) for name, (_, markers) in cases.items()]))

## 3. Transform each record independently

The default single centred mean uses five native positions separately in each
phase, restores original missing positions, and precedes median bin aggregation.
The default candidate grids are 0–100 %Wpeak in steps of 1 and 0–180 s in steps
of 5. At least two occupied bins are required; interpolation stays within their
bounds. Amplitudes retain their input units.

The supplied recovery anchors are audited synthetic markers. For the truncated
record `None` invokes the default search strictly after peak, which finds no
transition and retains an unavailable recovery.

In [ ]:
results = {name: transform_record(record, markers, channels=["VCO2"])
           for name, (record, markers) in cases.items()}
for phase in ("ramp", "recovery"):
    print(phase, getattr(results["ramp_4min"], phase)["VCO2"].grid)

## 4. Inspect matrices, counts and support

Each exported matrix has named records as rows and phase coordinates as columns.
`defined` marks available output, `observed_bin` marks occupied-bin summaries,
and `interpolated` marks internal estimates. The latter two partition `defined`.
Occupied summaries already include smoothing and aggregation; interpolation does
not create independent observations. No maximum gap length is imposed.

The ramp begins at 20%: the first 20 candidate locations remain unavailable.
Recovery has deliberate internal gaps and missing terminal values, making its
actual support shorter than the candidate grid. Each record retains its own mask.

In [ ]:
def export_results(cases, results, output_dir):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows, support, metadata = [], [], []
    for name, (record, markers) in cases.items():
        record.to_csv(out / f'native_{name}.csv', index=False)
        metadata.append(dict(record=name, **asdict(results[name].markers),
                             ramp_duration_s=markers.peak_time_s - markers.ramp_start_s,
                             ramp_slope_w_per_min=.8 * markers.peak_workload_w /
                             (markers.peak_time_s - markers.ramp_start_s) * 60,
                             ramp_observations=int((record.time_s <= markers.peak_time_s).sum()),
                             recovery_observations=int((record.time_s > markers.peak_time_s).sum())))
        for phase in ('ramp', 'recovery'):
            c = getattr(results[name], phase)['VCO2']
            support.append(dict(record=name, phase=phase, grid_size=len(c.grid),
                                defined=int(c.defined.sum()), observed_bin=int(c.observed_bin.sum()),
                                interpolated=int(c.interpolated.sum())))
            rows.extend(dict(record=name, phase=phase, channel='VCO2', coordinate=float(g),
                             value=float(v), defined=bool(d), observed_bin=bool(o), interpolated=bool(i))
                        for g, v, d, o, i in zip(c.grid, c.values, c.defined, c.observed_bin, c.interpolated))
    pd.DataFrame(rows).to_csv(out / 'trajectories.csv', index=False)
    pd.DataFrame(metadata).to_csv(out / 'markers_and_counts.csv', index=False)
    pd.DataFrame(support).to_csv(out / 'support.csv', index=False)
    # Every matrix uses the same named row order and explicit coordinate columns.
    for phase in ('ramp', 'recovery'):
        grid = getattr(next(iter(results.values())), phase)['VCO2'].grid
        for field in ('values', 'defined', 'observed_bin', 'interpolated'):
            matrix = pd.DataFrame([getattr(getattr(r, phase)['VCO2'], field) for r in results.values()],
                                  index=list(results), columns=grid)
            matrix.index.name = 'record'
            matrix.to_csv(out / f'{phase}_{field}.csv')
    paired, pair_rows = {}, []
    for phase in ('ramp', 'recovery'):
        paired[phase] = {}
        for label, first, second in [('alignment', 'ramp_4min', 'ramp_8min'),
                                     ('known_offset', 'ramp_8min', 'ramp_8min_plus30'),
                                     ('missing_recovery', 'ramp_8min', 'ramp_8min_no_recovery')]:
            pair = compare_records(results[first], results[second], phase=phase, channel='VCO2')
            paired[phase][label] = dict(n_supported=pair.n_supported, n_minimum=pair.n_minimum,
                                       rms=float(pair.rms) if np.isfinite(pair.rms) else None)
            pair_rows.extend(dict(case=label, phase=phase, coordinate=float(g), difference=float(v),
                                  jointly_defined=bool(d))
                             for g, v, d in zip(pair.grid, pair.pointwise_difference, pair.jointly_defined))
    pd.DataFrame(pair_rows).to_csv(out / 'paired_differences.csv', index=False)
    summary = dict(package_version=version('cpet-functional-skeleton'), data='synthetic only',
                   seeds=[401, 801], channel_units={'VCO2':'mL/min'},
                   coordinate_units={'ramp':'%Wpeak', 'recovery':'seconds from anchor'},
                   rolling_window=5, paired=paired)
    (out / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    return pd.DataFrame(metadata), pd.DataFrame(support), summary

output_dir = Path("cpet_tutorial_output")
metadata, support, summary = export_results(cases, results, output_dir)
display(metadata)
display(support)
display(pd.read_csv(output_dir / "ramp_values.csv", index_col="record"))
display(pd.read_csv(output_dir / "recovery_values.csv", index_col="record"))

## 5. Give ramp and recovery the same visual inspection

Left: native phase time and observations. Right: represented coordinates and
the analytic target. Both plots retain original channel units. The mask plots
distinguish occupied bins, interpolation and unavailable positions for all cases.

In [ ]:
def plot_results(cases, results, output_dir):
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    out = Path(output_dir)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for name in ('ramp_4min', 'ramp_8min'):
        record, markers = cases[name]
        for row, phase in enumerate(('ramp', 'recovery')):
            part = record.loc[record.time_s <= markers.peak_time_s] if phase == 'ramp' else record.loc[record.time_s >= markers.recovery_anchor_s]
            native_time = part.time_s - (markers.ramp_start_s if phase == 'ramp' else markers.recovery_anchor_s)
            axes[row, 0].plot(native_time, part.VCO2, '.-', label=name, markersize=3)
            curve = getattr(results[name], phase)['VCO2']
            axes[row, 1].plot(curve.grid, curve.values, label=name)
    for row, phase in enumerate(('ramp', 'recovery')):
        grid = getattr(results['ramp_4min'], phase)['VCO2'].grid
        axes[row, 1].plot(grid, analytic_response(grid, phase), 'k--', label='analytic target')
        for col in (0, 1):
            axes[row, col].set_ylabel('Synthetic VCO2 (mL/min)')
            axes[row, col].set_title(f'{phase.capitalize()}: ' + ('native observations' if col == 0 else 'represented'))
            axes[row, col].legend()
        axes[row, 0].set_xlabel('Seconds from ' + ('ramp onset' if phase == 'ramp' else 'recovery anchor'))
        axes[row, 1].set_xlabel('%Wpeak' if phase == 'ramp' else 'Seconds from recovery anchor')
    fig.savefig(out / 'native_and_represented.png', dpi=140)
    mask_fig, mask_axes = plt.subplots(2, 1, figsize=(12, 5), constrained_layout=True)
    colors = ['#dedede', '#2166ac', '#f4a340']
    for ax, phase in zip(mask_axes, ('ramp', 'recovery')):
        curves = [getattr(r, phase)['VCO2'] for r in results.values()]
        states = np.array([c.observed_bin.astype(int) + 2 * c.interpolated for c in curves])
        ax.imshow(states, interpolation='nearest', aspect='auto', cmap=ListedColormap(colors), vmin=0, vmax=2)
        ax.set_yticks(range(len(results)), labels=list(results))
        ticks = np.linspace(0, len(curves[0].grid)-1, 6, dtype=int)
        ax.set_xticks(ticks, labels=curves[0].grid[ticks])
        ax.set_xlabel('Ramp %Wpeak' if phase == 'ramp' else 'Recovery seconds from anchor')
    mask_axes[0].legend(handles=[Patch(color=c, label=l) for c, l in zip(colors, ['unavailable', 'occupied bin', 'interpolated'])],
                        loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=3)
    mask_fig.savefig(out / 'support_masks.png', dpi=140)
    return fig, mask_fig

plot_results(cases, results, output_dir)
plt.show()

## 6. Optional paired differences

Pairing is a downstream choice after independent transformation. Values are
second minus first on joint support. RMS eligibility uses at least 21 of 101
ramp locations or 8 of 37 recovery locations by default. The known-offset case
must retain 30 mL/min wherever both curves are defined. The alignment example
need not have zero RMS; sampling and smoothing differ.

In [ ]:
for phase in ("ramp", "recovery"):
    known = compare_records(results["ramp_8min"], results["ramp_8min_plus30"], phase=phase, channel="VCO2")
    np.testing.assert_allclose(known.pointwise_difference[known.jointly_defined], 30., atol=1e-10)
    print(phase, summary["paired"][phase])
missing = results["ramp_8min_no_recovery"].recovery["VCO2"]
assert not missing.defined.any() and np.isnan(missing.values).all()
assert results["ramp_8min"].recovery["VCO2"].defined.any()
for result in results.values():
    for phase in ("ramp", "recovery"):
        curve = getattr(result, phase)["VCO2"]
        np.testing.assert_array_equal(curve.defined, curve.observed_bin | curve.interpolated)
        assert not (curve.observed_bin & curve.interpolated).any()
display(pd.read_csv(output_dir / "paired_differences.csv").head())

## 7. Download and continue

The ZIP contains synthetic native inputs, audited markers/counts, long-form
trajectories, both phase matrices, all three masks, paired differences, summary
metadata and figures. Empty CSV numerical cells mean unavailable. Re-running
replaces the generated files in this relative output folder.

The GitHub repository separately provides [the historical reconstruction
materials](https://github.com/mlacasa/CPET_method/tree/v0.2.1/validation), with
their dependencies and limitations. This tutorial supplies none of the original
clinical sampling templates and does not reproduce the study's error estimates.

In [ ]:
def archive_results(output_dir):
    out = Path(output_dir)
    # Build and verify on a local temporary filesystem before copying to cloud drives.
    with tempfile.TemporaryDirectory() as temp:
        archive = Path(shutil.make_archive(str(Path(temp) / 'cpet_synthetic_results'), 'zip', out))
        with zipfile.ZipFile(archive) as handle:
            assert handle.testzip() is None
        target = out.parent / (out.name + '.zip')
        shutil.copyfile(archive, target)
    return target

archive = archive_results(output_dir)
display(FileLink(str(archive)))
try:
    from google.colab import files
except ImportError:
    print("Local execution: use the ZIP link above.")
else:
    files.download(str(archive))